# Modeling

In [1]:
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy='prior')

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path.cwd().parent / "data" / "processed"

train = pd.read_parquet(DATA_DIR / "features_train_refined.parquet")
test = pd.read_parquet(DATA_DIR / "features_test_refined.parquet")

print(train.shape)
print(test.shape)

(307511, 203)
(48744, 202)


In [3]:
from IPython.display import display
display(train.columns.to_list())

['SK_ID_CURR',
 'TARGET',
 'NAME_CONTRACT_TYPE',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'CNT_CHILDREN',
 'AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'REGION_POPULATION_RELATIVE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'OWN_CAR_AGE',
 'FLAG_MOBIL',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_CONT_MOBILE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'OCCUPATION_TYPE',
 'CNT_FAM_MEMBERS',
 'REGION_RATING_CLIENT',
 'REGION_RATING_CLIENT_W_CITY',
 'WEEKDAY_APPR_PROCESS_START',
 'HOUR_APPR_PROCESS_START',
 'REG_REGION_NOT_LIVE_REGION',
 'REG_REGION_NOT_WORK_REGION',
 'LIVE_REGION_NOT_WORK_REGION',
 'REG_CITY_NOT_LIVE_CITY',
 'REG_CITY_NOT_WORK_CITY',
 'LIVE_CITY_NOT_WORK_CITY',
 'ORGANIZATION_TYPE',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'APARTMENTS_AVG',
 'BASEMENTAREA_AVG',
 'YEARS_BEGINEXPLUATATION_A

In [4]:
ID_COL ="SK_ID_CURR"
TARGET_COL = "TARGET"

y = train[TARGET_COL].copy()

X = train.drop(columns=[TARGET_COL, ID_COL]).copy()
X_test = test.drop(columns=[ID_COL]).copy()

test_ids = test[ID_COL].copy()

print(X.shape)
print(y.shape)
print(X_test.shape)

print("\nTarget distribution:")
print(y.value_counts(normalize=True))

(307511, 201)
(307511,)
(48744, 201)

Target distribution:
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


In [5]:
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

print("\nCategorical columns:")
print(categorical_cols)

Numeric columns: 185
Categorical columns: 16

Categorical columns:
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


In [6]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [8]:
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy='most_frequent')),
        ("one_hot", OneHotEncoder(
            handle_unknown="ignore"
        ))
    ]
)

In [9]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy='median')),
        ("scaler", StandardScaler())
    ]
)

In [10]:
preprocessor = ColumnTransformer(
    transformers=(
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols)
    )
)

In [11]:
logistic_model = LogisticRegression(
    max_iter=1000,
    solver='liblinear'
)

In [12]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", logistic_model)
    ]
)

In [13]:
logreg_fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]
    
    logistic_pipeline.fit(X_train, y_train)
    
    valid_pred = logistic_pipeline.predict_proba(X_valid)[:, 1]
    
    auc = roc_auc_score(y_valid, valid_pred)
    logreg_fold_scores.append(auc)
    
    print(f"Fold {fold}: AUC = {auc:.5f}")

Fold 1: AUC = 0.76291
Fold 2: AUC = 0.76896
Fold 3: AUC = 0.76928
Fold 4: AUC = 0.77051
Fold 5: AUC = 0.76238


In [14]:
print()
print(f"Mean CV AUC: {np.mean(logreg_fold_scores):.5f}")
print(f"Std CV AUC: {np.std(logreg_fold_scores):.5f}")


Mean CV AUC: 0.76681
Std CV AUC: 0.00344


In [15]:
results = []

results.append({
    "experiment": "B01",
    "model": "Logistic Regression",
    "features": "All current feautres",
    "mean_cv_auc": np.mean(logreg_fold_scores),
    "std_cv_auc": np.std(logreg_fold_scores)
})

results_df = pd.DataFrame(results)
display(results_df)

,experiment,model,features,mean_cv_auc,std_cv_auc
0,B01,Logistic Regression,All current feautres,0.766807,0.003442


# LightGBM

In [9]:
from lightgbm import (
    LGBMClassifier,
    early_stopping,
    log_evaluation,
)

In [10]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
import numpy as np

In [11]:
categorical_pipeline_tree = Pipeline([
    ("imputer", SimpleImputer(strategy='constant', fill_value="__Missing__")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

In [12]:
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_cols),
        ("cat", categorical_pipeline_tree, categorical_cols)
    ],
    sparse_threshold=1.0
)

In [13]:
lgbm_params = {
    "objective": "binary",
    "n_estimators": 5000,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": -1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
    "random_state": 42,
    "n_jobs": -1,
}

In [21]:
lgbm_fold_scores = []

oof_lgbm = np.zeros(len(X))
test_lgbm = np.zeros(len(X_test))

In [22]:
for fold, (train_idx, valid_idx) in enumerate(cv.split(X,y), start=1):
    print(f"\n =====Fold===== {fold}")
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]
    
    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]
    
    # Fit preprocessing on only this training fold
    fold_preprocessor = clone(tree_preprocessor)
    
    X_train_t = fold_preprocessor.fit_transform(X_train)
    X_valid_t = fold_preprocessor.transform(X_valid)
    X_test_t = fold_preprocessor.transform(X_test)
    
    model = LGBMClassifier(**lgbm_params)
    
    model.fit(
        X_train_t,
        y_train,
        eval_set = [(X_valid_t, y_valid)],
        eval_metric= "auc",
        callbacks=[
            early_stopping(200),
            log_evaluation(0)
        ]
    )
    
    valid_pred = model.predict_proba(X_valid_t, num_iteration=model.best_iteration_)[:,1]
    
    oof_lgbm[valid_idx] = valid_pred
    
    test_lgbm += (
        model.predict_proba(
            X_test_t,
            num_iteration=model.best_iteration_)[:, 1] / cv.n_splits
        
    ) 
    fold_auc = roc_auc_score(y_valid, valid_pred)
    lgbm_fold_scores.append(fold_auc)
    
    print(
        f"Fold {fold} AUC: {fold_auc:.5f} | ",
        f"Best Iteration: {model.best_iteration_}"
    )


 =====Fold===== 1


.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.072486 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23910
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 321
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432482
[LightGBM] [Info] Start training from score -2.432482
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[789]	valid_0's auc: 0.778954	valid_0's binary_logloss: 0.239355
Fold 1 AUC: 0.77895 |  Best Iteration: 789

 =====Fold===== 2


.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.066269 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23959
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 321
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[1212]	valid_0's auc: 0.787553	valid_0's binary_logloss: 0.236708
Fold 2 AUC: 0.78755 |  Best Iteration: 1212

 =====Fold===== 3


...

[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.071726 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23891
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 320
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[1169]	valid_0's auc: 0.782192	valid_0's binary_logloss: 0.238811
Fold 3 AUC: 0.78219 |  Best Iteration: 1169

 =====Fold===== 4


.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.073544 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23919
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 321
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[1023]	valid_0's auc: 0.78603	valid_0's binary_logloss: 0.237226
Fold 4 AUC: 0.78603 |  Best Iteration: 1023

 =====Fold===== 5


.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.071109 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23904
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 321
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[1101]	valid_0's auc: 0.778661	valid_0's binary_logloss: 0.240118
Fold 5 AUC: 0.77866 |  Best Iteration: 1101


In [23]:
print("\nLightGBM Results")
print("----------------")

print(
    f"Mean fold AUC : "
    f"{np.mean(lgbm_fold_scores):.5f}"
)

print(
    f"Std fold AUC  : "
    f"{np.std(lgbm_fold_scores):.5f}"
)

print(
    f"OOF AUC       : "
    f"{roc_auc_score(y, oof_lgbm):.5f}"
)


LightGBM Results
----------------
Mean fold AUC : 0.78268
Std fold AUC  : 0.00361
OOF AUC       : 0.78267


In [24]:
np.mean(lgbm_fold_scores)

np.float64(0.7826779892480731)

In [25]:
roc_auc_score(y, oof_lgbm)

0.7826704916914884

In [26]:
results.append({
    "experiment": "B02",
    "model": "LightGBM",
    "features": "All current features",
    "mean_cv_auc": np.mean(lgbm_fold_scores),
    "std_cv_auc": np.std(lgbm_fold_scores),
})

results_df = pd.DataFrame(results)

display(results_df)

,experiment,model,features,mean_cv_auc,std_cv_auc
0,B01,Logistic Regression,All current feautres,0.766807,0.003442
1,B02,LightGBM,All current features,0.782678,0.003612


In [27]:
comparison = pd.DataFrame({
    "logreg_auc": logreg_fold_scores,
    "lgbm_auc": lgbm_fold_scores
})

comparison["gain"] = (
    comparison["lgbm_auc"]
    - comparison["logreg_auc"]
)

display(comparison)

print("Mean gain:", comparison["gain"].mean())

,logreg_auc,lgbm_auc,gain
0,0.762908,0.778954,0.016046
1,0.768963,0.787553,0.018590
2,0.769278,0.782192,0.012913
3,0.770506,0.786030,0.015524
4,0.762379,0.778661,0.016282


Mean gain: 0.015871189117555827


In [14]:
APPLICATION_FEATURES = [
    "DAYS_EMPLOYED_ANOMALY",
    "DAYS_EMPLOYED_CLEAN",
    "AGE_YEARS",
    "EMPLOYED_YEARS",
    "INCOME_CREDIT_RATIO",
    "ANNUITY_CREDIT_RATIO",
    "CREDIT_GOODS_RATIO",
    "INCOME_PER_PERSON",
    "EXT_SOURCE_MEAN",
    "EXT_SOURCE_COUNT",
    "EXT_SOURCE_MISSING_COUNT",
]

BUREAU_FEATURES = [
    "BUREAU_RECORD_COUNT",
    "HAS_BUREAU_HISTORY",
    "HAS_MICROLOAN",
    "BUREAU_ACTIVE_COUNT",
    "BUREAU_CLOSED_COUNT",
    "BUREAU_ACTIVE_SHARE",
    "BUREAU_CLOSED_SHARE",
    "BUREAU_CREDIT_SUM",
    "BUREAU_DEBT_SUM",
    "BUREAU_OVERDUE_SUM",
    "BUREAU_MAX_OVERDUE",
    "BUREAU_DEBT_CREDIT_RATIO",
    "BUREAU_DAYS_CREDIT_MEAN",
    "BUREAU_DAYS_CREDIT_MAX",
    "BUREAU_MICROLOAN_COUNT",
]

BUREAU_BAL_FEATURES = [
    "BUREAU_BAL_MONTH_COUNT",
    "HAS_BUREAU_BAL_HISTORY",
    "BUREAU_BAL_EVER_DELINQUENT",
    "BUREAU_BAL_DELINQUENT_SHARE",
    "BUREAU_BAL_WORST_STATUS",
]

PREVIOUS_APPLICATION_FEATURES = [
    "PREVIOUS_APPLICATION_RECORD_COUNT",
    "HAS_PREVIOUS_APPLICATION_HISTORY",
    "HAS_REFUSED_PRIOR",
    "PREV_APPROVED_COUNT",
    "PREV_REFUSED_COUNT",
    "PREV_APPROVED_SHARE",
    "PREV_REFUSED_SHARE",
    "PREV_APPLICATION_AMT_MEAN",
    "PREV_APPLICATION_AMT_MAX",
    "PREV_CREDIT_AMT_MEAN",
    "PREV_CREDIT_AMT_MAX",
    "PREV_APPLICATION_CREDIT_RATIO_MEAN",
    "PREV_DAYS_DECISION_MEAN",
    "PREV_DAYS_DECISION_MAX",
]

INSTALLMENT_FEATURES = [
    "INSTALLMENT_RECORD_COUNT",
    "HAS_INSTALLMENT_HISTORY",
    "INSTALLMENT_MEAN_DPD",
    "INSTALLMENT_MAX_DPD",
    "INSTALLMENT_MEAN_DBD",
    "INSTALLMENT_LATE_SHARE",
    "INSTALLMENT_MEAN_PAYMENT_RATIO",
    "INSTALLMENT_MEAN_SHORTFALL_RATIO",
    "INSTALLMENT_UNDERPAID_SHARE",
    "ANY_LATE_INSTALLMENT",
    "ANY_INSTALLMENT_SHORTFALL",
]

POS_FEATURES = [
    "POS_RECORD_COUNT",
    "HAS_POS_HISTORY",
    "POS_DPD_MONTHS",
    "POS_DPD_MAX",
    "POS_DPD_SHARE",
    "POS_DPD_DEF_MONTHS",
    "POS_DPD_DEF_MAX",
    "POS_DPD_DEF_SHARE",
    "ANY_POS_DPD",
    "ANY_POS_DPD_DEF",
]

CARD_FEATURES = [
    "CARD_RECORD_COUNT",
    "HAS_CARD_HISTORY",
    "CARD_BALANCE_MEAN",
    "CARD_UTIL_MEAN",
    "CARD_UTIL_MAX",
    "CARD_OVER_LIMIT_MONTHS",
    "CARD_OVER_LIMIT_SHARE",
    "CARD_DPD_MONTHS",
    "CARD_DPD_MAX",
    "CARD_DPD_DEF_MAX",
    "CARD_DPD_SHARE",
    "EVER_OVER_CARD_LIMIT",
    "ANY_CARD_DPD",
]

CROSS_TABLE_FEATURES = [
    "BUREAU_DEBT_INCOME_RATIO",
    "BUREAU_CREDIT_INCOME_RATIO",
]

In [15]:
ENGINEERED_GROUPS = {
    "application_engineered": APPLICATION_FEATURES,
    "bureau": BUREAU_FEATURES,
    "bureau_balance": BUREAU_BAL_FEATURES,
    "previous_application": PREVIOUS_APPLICATION_FEATURES,
    "installments": INSTALLMENT_FEATURES,
    "pos": POS_FEATURES,
    "credit_card": CARD_FEATURES,
    "cross_table": CROSS_TABLE_FEATURES,
}

ENGINEERED_FEATURES = [
    feature
    for group in ENGINEERED_GROUPS.values()
    for feature in group
]

print(len(ENGINEERED_FEATURES))

81


In [16]:
raw_application_cols = [
    col for col in X.columns
    if col not in ENGINEERED_FEATURES
]

print("Raw application features:", len(raw_application_cols))
print("Engineered features:", len(ENGINEERED_FEATURES))
print("All model features:", X.shape[1])

Raw application features: 120
Engineered features: 81
All model features: 201


In [17]:
assert (
    len(raw_application_cols)
    + len(ENGINEERED_FEATURES)
    == X.shape[1]
)

In [18]:

feature_experiments = {}

current = raw_application_cols.copy()
feature_experiments["F00_raw_application"] = current.copy()

current += APPLICATION_FEATURES
feature_experiments["F01_plus_app_engineered"] = current.copy()

current += BUREAU_FEATURES
feature_experiments["F02_plus_bureau"] = current.copy()

current += BUREAU_BAL_FEATURES
feature_experiments["F03_plus_bureau_balance"] = current.copy()

current += PREVIOUS_APPLICATION_FEATURES
feature_experiments["F04_plus_previous"] = current.copy()

current += INSTALLMENT_FEATURES
feature_experiments["F05_plus_installments"] = current.copy()

current += POS_FEATURES
feature_experiments["F06_plus_pos"] = current.copy()

current += CARD_FEATURES
feature_experiments["F07_plus_credit_card"] = current.copy()

current += CROSS_TABLE_FEATURES
feature_experiments["F08_plus_cross_table"] = current.copy()


# NEW: preserve the exact column order used in X
def preserve_feature_order(features, reference_columns):
    selected = set(features)

    return [
        col
        for col in reference_columns
        if col in selected
    ]


feature_experiments = {
    name: preserve_feature_order(cols, X.columns)
    for name, cols in feature_experiments.items()
}

In [19]:
for name, cols in feature_experiments.items():
    print(f"{name:28s} {len(cols):3d} features")

F00_raw_application          120 features
F01_plus_app_engineered      131 features
F02_plus_bureau              146 features
F03_plus_bureau_balance      151 features
F04_plus_previous            165 features
F05_plus_installments        176 features
F06_plus_pos                 186 features
F07_plus_credit_card         199 features
F08_plus_cross_table         201 features


In [20]:
assert (
    feature_experiments["F08_plus_cross_table"]
    == X.columns.tolist()
)

In [21]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

from lightgbm import LGBMClassifier, early_stopping, log_evaluation

import numpy as np

In [22]:
def make_tree_preprocessor(X_subset):

    numeric_cols = (
        X_subset
        .select_dtypes(include="number")
        .columns
        .tolist()
    )

    categorical_cols = (
        X_subset
        .select_dtypes(exclude="number")
        .columns
        .tolist()
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__"
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", "passthrough", numeric_cols),
            ("cat", categorical_pipeline, categorical_cols),
        ],
        sparse_threshold=1.0,
    )

    return preprocessor

In [23]:
def evaluate_lgbm_feature_set(
    X_subset,
    y,
    cv,
    lgbm_params,
):

    fold_scores = []
    best_iterations = []

    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X_subset, y),
        start=1,
    ):

        X_train = X_subset.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_valid = X_subset.iloc[valid_idx]
        y_valid = y.iloc[valid_idx]

        preprocessor = make_tree_preprocessor(X_train)

        X_train_t = preprocessor.fit_transform(X_train)
        X_valid_t = preprocessor.transform(X_valid)

        model = LGBMClassifier(**lgbm_params)

        model.fit(
            X_train_t,
            y_train,
            eval_set=[(X_valid_t, y_valid)],
            eval_metric="auc",
            callbacks=[
                early_stopping(200, verbose=False),
                log_evaluation(0),
            ],
        )

        valid_pred = model.predict_proba(
            X_valid_t,
            num_iteration=model.best_iteration_,
        )[:, 1]

        auc = roc_auc_score(y_valid, valid_pred)

        fold_scores.append(auc)
        best_iterations.append(model.best_iteration_)

        print(
            f"    Fold {fold}: "
            f"AUC={auc:.5f}, "
            f"best_iter={model.best_iteration_}"
        )

    return {
        "mean_cv_auc": np.mean(fold_scores),
        "std_cv_auc": np.std(fold_scores),
        "fold_scores": fold_scores,
        "mean_best_iteration": np.mean(best_iterations),
    }

In [91]:
feature_results = []

for experiment, cols in feature_experiments.items():

    print(f"\n{'=' * 60}")
    print(experiment)
    print(f"Number of features: {len(cols)}")
    print("=" * 60)

    result = evaluate_lgbm_feature_set(
        X[cols],
        y,
        cv,
        lgbm_params,
    )

    feature_results.append({
        "experiment": experiment,
        "n_features": len(cols),
        "mean_cv_auc": result["mean_cv_auc"],
        "std_cv_auc": result["std_cv_auc"],
        "mean_best_iteration": result["mean_best_iteration"],
    })


F00_raw_application
Number of features: 120


.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


    Fold 1: AUC=0.75524, best_iter=553


.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


    Fold 2: AUC=0.76655, best_iter=858


...

    Fold 3: AUC=0.78219, best_iter=1169


.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


    Fold 4: AUC=0.78603, best_iter=1023


.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


    Fold 5: AUC=0.77866, best_iter=1101


In [92]:
feature_results_df = pd.DataFrame(feature_results)

feature_results_df["auc_gain"] = (
    feature_results_df["mean_cv_auc"]
    .diff()
)

display(feature_results_df)

,experiment,n_features,mean_cv_auc,std_cv_auc,mean_best_iteration,auc_gain
0,F00_raw_application,120,0.759494,0.004727,630.2,NaN
1,F01_plus_app_engineered,131,0.767294,0.004060,847.4,0.007800
2,F02_plus_bureau,146,0.772805,0.003969,933.4,0.005511
3,F03_plus_bureau_balance,151,0.772963,0.003309,952.8,0.000158
4,F04_plus_previous,165,0.776137,0.003851,1021.0,0.003174
5,F05_plus_installments,176,0.779928,0.003233,909.4,0.003792
6,F06_plus_pos,186,0.780639,0.003308,1127.4,0.000711
7,F07_plus_credit_card,199,0.783043,0.003204,1183.4,0.002404
8,F08_plus_cross_table,201,0.782678,0.003612,1058.8,-0.000365


In [24]:
FULL_FEATURES = X.columns.tolist()

drop_experiments = {
    "A00_full": FULL_FEATURES,

    "A01_minus_app_engineered": [
        c for c in FULL_FEATURES
        if c not in APPLICATION_FEATURES
    ],

    "A02_minus_bureau": [
        c for c in FULL_FEATURES
        if c not in BUREAU_FEATURES
    ],

    "A03_minus_bureau_balance": [
        c for c in FULL_FEATURES
        if c not in BUREAU_BAL_FEATURES
    ],

    "A04_minus_previous": [
        c for c in FULL_FEATURES
        if c not in PREVIOUS_APPLICATION_FEATURES
    ],

    "A05_minus_installments": [
        c for c in FULL_FEATURES
        if c not in INSTALLMENT_FEATURES
    ],

    "A06_minus_pos": [
        c for c in FULL_FEATURES
        if c not in POS_FEATURES
    ],

    "A07_minus_credit_card": [
        c for c in FULL_FEATURES
        if c not in CARD_FEATURES
    ],

    "A08_minus_cross_table": [
        c for c in FULL_FEATURES
        if c not in CROSS_TABLE_FEATURES
    ],
}

In [25]:
for name, cols in drop_experiments.items():
    print(f"{name:28s} {len(cols):3d}")

A00_full                     201
A01_minus_app_engineered     190
A02_minus_bureau             186
A03_minus_bureau_balance     196
A04_minus_previous           187
A05_minus_installments       190
A06_minus_pos                191
A07_minus_credit_card        188
A08_minus_cross_table        199


In [53]:
ablation_results = []

for experiment, cols in drop_experiments.items():

    print(f"\n{'=' * 60}")
    print(experiment)
    print(f"Number of features: {len(cols)}")
    print("=" * 60)

    result = evaluate_lgbm_feature_set(
        X[cols],
        y,
        cv,
        lgbm_params,
    )

    ablation_results.append({
        "experiment": experiment,
        "n_features": len(cols),
        "mean_cv_auc": result["mean_cv_auc"],
        "std_cv_auc": result["std_cv_auc"],
    })


A00_full
Number of features: 201


.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.059541 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23910
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 321
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432482
[LightGBM] [Info] Start training from score -2.432482
    Fold 1: AUC=0.77895, best_iter=789


.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.060799 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23959
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 321
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486
    Fold 2: AUC=0.78755, best_iter=1212


...

[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.054893 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23381
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 318
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486
    Fold 3: AUC=0.78241, best_iter=1258


.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.056730 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23409
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 319
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486
    Fold 4: AUC=0.78583, best_iter=1014


.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.060620 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23394
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 319
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486
    Fold 5: AUC=0.78032, best_iter=1368


In [54]:
ablation_df = pd.DataFrame(ablation_results)

full_auc = ablation_df.loc[
    ablation_df["experiment"] == "A00_full",
    "mean_cv_auc"
].iloc[0]

ablation_df["auc_loss_when_removed"] = (
    full_auc - ablation_df["mean_cv_auc"]
)

display(
    ablation_df.sort_values(
        "auc_loss_when_removed",
        ascending=False
    )
)

,experiment,n_features,mean_cv_auc,std_cv_auc,auc_loss_when_removed
1,A01_minus_app_engineered,190,0.779191,0.003018,0.003487
4,A04_minus_previous,187,0.779582,0.002961,0.003096
2,A02_minus_bureau,186,0.780249,0.003224,0.002429
7,A07_minus_credit_card,188,0.780888,0.003166,0.001790
5,A05_minus_installments,190,0.781156,0.003122,0.001522
6,A06_minus_pos,191,0.782585,0.003193,0.000093
0,A00_full,201,0.782678,0.003612,0.000000
8,A08_minus_cross_table,199,0.783043,0.003204,-0.000365
3,A03_minus_bureau_balance,196,0.783167,0.003556,-0.000489


In [55]:
def get_cv_feature_importance(
    X,
    y,
    cv,
    lgbm_params,
):
    importance_frames = []

    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X, y),
        start=1,
    ):

        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_valid = X.iloc[valid_idx]
        y_valid = y.iloc[valid_idx]

        preprocessor = make_tree_preprocessor(X_train)

        X_train_t = preprocessor.fit_transform(X_train)
        X_valid_t = preprocessor.transform(X_valid)

        model = LGBMClassifier(**lgbm_params)

        model.fit(
            X_train_t,
            y_train,
            eval_set=[(X_valid_t, y_valid)],
            eval_metric="auc",
            callbacks=[
                early_stopping(200, verbose=False),
                log_evaluation(0),
            ],
        )

        feature_names = preprocessor.get_feature_names_out()

        importance = pd.DataFrame({
            "feature": feature_names,
            "gain": model.booster_.feature_importance(
                importance_type="gain"
            ),
            "fold": fold,
        })

        importance_frames.append(importance)

    return pd.concat(
        importance_frames,
        ignore_index=True
    )

In [56]:
cv_importance = get_cv_feature_importance(
    X,
    y,
    cv,
    lgbm_params,
)

.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.054022 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23910
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 321
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432482
[LightGBM] [Info] Start training from score -2.432482


.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.053065 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23959
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 321
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486


.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.067101 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23891
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 320
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486


.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.059189 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23919
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 321
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486


.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.061386 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23904
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 321
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486


In [61]:
importance_summary = (
    cv_importance
    .groupby("feature", as_index=False)
    .agg(
        mean_gain=("gain", "mean"),
        std_gain=("gain", "std"),
        folds_used=("gain", lambda x: (x > 0).sum()),
    )
    .sort_values(
        "mean_gain",
        ascending=False
    )
)

display(
    importance_summary.head(30)
)

,feature,mean_gain,std_gain,folds_used
225,num__EXT_SOURCE_MEAN,171034.745977,12152.066510,5
223,num__EXT_SOURCE_3,26369.735292,6846.458843,5
157,num__ANNUITY_CREDIT_RATIO,23259.004885,1208.767437,5
222,num__EXT_SOURCE_2,18375.831393,4499.512637,5
203,num__CREDIT_GOODS_RATIO,15282.446284,835.856776,5
221,num__EXT_SOURCE_1,13371.281339,2241.732938,5
181,num__BUREAU_DEBT_CREDIT_RATIO,12085.791205,776.669875,5
308,num__PREV_APPLICATION_CREDIT_RATIO_MEAN,11935.856346,1466.499837,5
270,num__INSTALLMENT_LATE_SHARE,11691.067262,1306.921742,5
147,num__AMT_ANNUITY,10692.165186,844.602146,5


In [62]:
importance_summary["gain_pct"] = (
    100
    * importance_summary["mean_gain"]
    / importance_summary["mean_gain"].sum()
)

display(
    importance_summary[
        ["feature", "mean_gain", "gain_pct", "folds_used"]
    ].head(30)
)

,feature,mean_gain,gain_pct,folds_used
225,num__EXT_SOURCE_MEAN,171034.745977,24.852522,5
223,num__EXT_SOURCE_3,26369.735292,3.831703,5
157,num__ANNUITY_CREDIT_RATIO,23259.004885,3.379693,5
222,num__EXT_SOURCE_2,18375.831393,2.670134,5
203,num__CREDIT_GOODS_RATIO,15282.446284,2.220644,5
221,num__EXT_SOURCE_1,13371.281339,1.942939,5
181,num__BUREAU_DEBT_CREDIT_RATIO,12085.791205,1.756148,5
308,num__PREV_APPLICATION_CREDIT_RATIO_MEAN,11935.856346,1.734362,5
270,num__INSTALLMENT_LATE_SHARE,11691.067262,1.698792,5
147,num__AMT_ANNUITY,10692.165186,1.553645,5


In [26]:
PERMUTATION_FEATURES = [
    "EXT_SOURCE_MEAN",
    "EXT_SOURCE_3",
    "ANNUITY_CREDIT_RATIO",
    "EXT_SOURCE_2",
    "CREDIT_GOODS_RATIO",
    "EXT_SOURCE_1",
    "BUREAU_DEBT_CREDIT_RATIO",
    "PREV_APPLICATION_CREDIT_RATIO_MEAN",
    "INSTALLMENT_LATE_SHARE",
    "AMT_ANNUITY",
    "BUREAU_DAYS_CREDIT_MAX",
    "DAYS_EMPLOYED_CLEAN",
    "CARD_OVER_LIMIT_SHARE",
    "PREV_DAYS_DECISION_MEAN",
    "POS_RECORD_COUNT",
    "DAYS_BIRTH",
    "PREV_REFUSED_SHARE",
    "BUREAU_DAYS_CREDIT_MEAN",
    "DAYS_ID_PUBLISH",
    "EMPLOYED_YEARS",
    "BUREAU_CREDIT_SUM",
    "INCOME_CREDIT_RATIO",
    "OWN_CAR_AGE",
    "AGE_YEARS",
    "BUREAU_MAX_OVERDUE",
    "DAYS_REGISTRATION",
    "INSTALLMENT_MEAN_DBD",
    "PREV_DAYS_DECISION_MAX",
    "PREV_CREDIT_AMT_MAX",
    "BUREAU_CREDIT_INCOME_RATIO",
]

In [27]:
def get_cv_permutation_importance(
    X,
    y,
    cv,
    lgbm_params,
    features,
    random_state=42,
):
    permutation_results = []

    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X, y),
        start=1,
    ):

        print(f"\n===== Fold {fold} =====")

        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_valid = X.iloc[valid_idx]
        y_valid = y.iloc[valid_idx]

        # Fit preprocessing only on training fold
        preprocessor = make_tree_preprocessor(X_train)

        X_train_t = preprocessor.fit_transform(X_train)
        X_valid_t = preprocessor.transform(X_valid)

        model = LGBMClassifier(**lgbm_params)

        model.fit(
            X_train_t,
            y_train,
            eval_set=[(X_valid_t, y_valid)],
            eval_metric="auc",
            callbacks=[
                early_stopping(200, verbose=False),
                log_evaluation(0),
            ],
        )

        # Normal validation performance
        baseline_pred = model.predict_proba(
            X_valid_t,
            num_iteration=model.best_iteration_,
        )[:, 1]

        baseline_auc = roc_auc_score(
            y_valid,
            baseline_pred,
        )

        print(f"Baseline AUC: {baseline_auc:.5f}")

        rng = np.random.default_rng(
            random_state + fold
        )

        # Shuffle one original feature at a time
        for feature in features:

            X_permuted = X_valid.copy()

            X_permuted[feature] = rng.permutation(
                X_permuted[feature].to_numpy()
            )

            X_permuted_t = preprocessor.transform(
                X_permuted
            )

            permuted_pred = model.predict_proba(
                X_permuted_t,
                num_iteration=model.best_iteration_,
            )[:, 1]

            permuted_auc = roc_auc_score(
                y_valid,
                permuted_pred,
            )

            auc_drop = (
                baseline_auc - permuted_auc
            )

            permutation_results.append({
                "fold": fold,
                "feature": feature,
                "baseline_auc": baseline_auc,
                "permuted_auc": permuted_auc,
                "auc_drop": auc_drop,
            })

    return pd.DataFrame(permutation_results)

In [65]:
permutation_df = get_cv_permutation_importance(
    X=X,
    y=y,
    cv=cv,
    lgbm_params=lgbm_params,
    features=PERMUTATION_FEATURES,
)


===== Fold 1 =====


.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.053640 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23910
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 321
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432482
[LightGBM] [Info] Start training from score -2.432482
Baseline AUC: 0.77895

===== Fold 2 =====


.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.049382 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23959
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 321
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486
Baseline AUC: 0.78755

===== Fold 3 =====


...

[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.049369 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23891
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 320
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486
Baseline AUC: 0.78219

===== Fold 4 =====


.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.056243 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23919
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 321
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486
Baseline AUC: 0.78603

===== Fold 5 =====


.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.051916 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23904
[LightGBM] [Info] Number of data points in the train set: 246009, number of used features: 321
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486
Baseline AUC: 0.77866


In [66]:
permutation_summary = (
    permutation_df
    .groupby("feature", as_index=False)
    .agg(
        mean_auc_drop=("auc_drop", "mean"),
        std_auc_drop=("auc_drop", "std"),
        min_auc_drop=("auc_drop", "min"),
        max_auc_drop=("auc_drop", "max"),
    )
    .sort_values(
        "mean_auc_drop",
        ascending=False,
    )
)

display(permutation_summary)

,feature,mean_auc_drop,std_auc_drop,min_auc_drop,max_auc_drop
19,EXT_SOURCE_MEAN,0.059835,0.006638,0.050380,0.066870
2,ANNUITY_CREDIT_RATIO,0.008357,0.001739,0.005627,0.009560
1,AMT_ANNUITY,0.004128,0.000713,0.003233,0.004654
16,EXT_SOURCE_1,0.003492,0.000394,0.003143,0.004107
10,CREDIT_GOODS_RATIO,0.003271,0.000236,0.003025,0.003595
25,PREV_APPLICATION_CREDIT_RATIO_MEAN,0.002774,0.000280,0.002429,0.003127
21,INSTALLMENT_LATE_SHARE,0.002414,0.000203,0.002195,0.002611
18,EXT_SOURCE_3,0.002394,0.000601,0.001637,0.003143
23,OWN_CAR_AGE,0.002192,0.000507,0.001780,0.002758
7,BUREAU_DEBT_CREDIT_RATIO,0.001870,0.000404,0.001469,0.002344


In [67]:
cv_data = []

for fold, (train_idx, valid_idx) in enumerate(
    cv.split(X, y),
    start=1
):
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    preprocessor = make_tree_preprocessor(X_train)

    X_train_t = preprocessor.fit_transform(X_train)
    X_valid_t = preprocessor.transform(X_valid)

    cv_data.append({
        "fold": fold,
        "X_train": X_train_t,
        "y_train": y_train,
        "X_valid": X_valid_t,
        "y_valid": y_valid,
    })

In [28]:
def evaluate_lgbm_params(params, cv_data):

    fold_scores = []
    best_iterations = []

    for data in cv_data:

        model = LGBMClassifier(**params)

        model.fit(
            data["X_train"],
            data["y_train"],
            eval_X=data["X_valid"],
            eval_y=data["y_valid"],
            eval_metric="auc",
            callbacks=[
                early_stopping(200, verbose=False),
                log_evaluation(0),
            ],
        )

        pred = model.predict_proba(
            data["X_valid"],
            num_iteration=model.best_iteration_,
        )[:, 1]

        auc = roc_auc_score(
            data["y_valid"],
            pred,
        )

        fold_scores.append(auc)
        best_iterations.append(model.best_iteration_)

    return {
        "mean_auc": np.mean(fold_scores),
        "std_auc": np.std(fold_scores),
        "fold_scores": fold_scores,
        "mean_best_iteration": np.mean(best_iterations),
    }

In [29]:
complexity_trials = [
    {
        "name": "T01_leaves_15",
        "num_leaves": 15,
        "min_child_samples": 20,
    },
    {
        "name": "T02_leaves_31",
        "num_leaves": 31,
        "min_child_samples": 20,
    },
    {
        "name": "T03_leaves_63",
        "num_leaves": 63,
        "min_child_samples": 20,
    },
    {
        "name": "T04_leaves_31_child50",
        "num_leaves": 31,
        "min_child_samples": 50,
    },
    {
        "name": "T05_leaves_31_child100",
        "num_leaves": 31,
        "min_child_samples": 100,
    },
]

In [30]:
base_params = {
    "objective": "binary",
    "n_estimators": 5000,
    "learning_rate": 0.03,

    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "reg_alpha": 0.1,
    "reg_lambda": 0.1,

    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}

In [71]:
tuning_results = []

for trial in complexity_trials:

    name = trial["name"]

    params = {
        **base_params,
        **{
            k: v
            for k, v in trial.items()
            if k != "name"
        },
    }

    result = evaluate_lgbm_params(
        params,
        cv_data,
    )

    tuning_results.append({
        "experiment": name,
        "num_leaves": params["num_leaves"],
        "min_child_samples": params["min_child_samples"],
        "mean_cv_auc": result["mean_auc"],
        "std_cv_auc": result["std_auc"],
        "mean_best_iteration": result["mean_best_iteration"],
    })

    print(
        f"{name}: "
        f"{result['mean_auc']:.6f} "
        f"+/- {result['std_auc']:.6f}"
    )

T01_leaves_15: 0.783226 +/- 0.003127
T02_leaves_31: 0.782678 +/- 0.003612
T03_leaves_63: 0.782411 +/- 0.002786
T04_leaves_31_child50: 0.783620 +/- 0.003472
T05_leaves_31_child100: 0.783652 +/- 0.003209


In [72]:
tuning_df = (
    pd.DataFrame(tuning_results)
    .sort_values("mean_cv_auc", ascending=False)
)

display(tuning_df)

,experiment,num_leaves,min_child_samples,mean_cv_auc,std_cv_auc,mean_best_iteration
4,T05_leaves_31_child100,31,100,0.783652,0.003209,1064.6
3,T04_leaves_31_child50,31,50,0.783620,0.003472,1140.4
0,T01_leaves_15,15,20,0.783226,0.003127,2013.8
1,T02_leaves_31,31,20,0.782678,0.003612,1058.8
2,T03_leaves_63,63,20,0.782411,0.002786,666.4


In [78]:
BASELINE_AUC = 0.782678

tuning_df["gain_vs_baseline"] = (
    tuning_df["mean_cv_auc"] - BASELINE_AUC
)

display(tuning_df)

,experiment,num_leaves,min_child_samples,mean_cv_auc,std_cv_auc,mean_best_iteration,gain_vs_baseline
4,T05_leaves_31_child100,31,100,0.783652,0.003209,1064.6,9.737784e-04
3,T04_leaves_31_child50,31,50,0.783620,0.003472,1140.4,9.420932e-04
0,T01_leaves_15,15,20,0.783226,0.003127,2013.8,5.481406e-04
1,T02_leaves_31,31,20,0.782678,0.003612,1058.8,-1.075193e-08
2,T03_leaves_63,63,20,0.782411,0.002786,666.4,-2.672158e-04


In [31]:
tuned_base_params = {
    "objective": "binary",
    "n_estimators": 5000,
    "learning_rate": 0.03,

    "num_leaves": 31,
    "min_child_samples": 50,

    "reg_alpha": 0.1,
    "reg_lambda": 0.1,

    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}

In [75]:
sampling_trials = [
    {
        "name": "S01_no_sampling",
        "colsample_bytree": 1.0,
        "subsample": 1.0,
        "subsample_freq": 0,
    },

    {
        "name": "S02_feature_08",
        "colsample_bytree": 0.8,
        "subsample": 1.0,
        "subsample_freq": 0,
    },

    {
        "name": "S03_feature_07",
        "colsample_bytree": 0.7,
        "subsample": 1.0,
        "subsample_freq": 0,
    },

    {
        "name": "S04_row_08",
        "colsample_bytree": 1.0,
        "subsample": 0.8,
        "subsample_freq": 1,
    },

    {
        "name": "S05_row_07",
        "colsample_bytree": 1.0,
        "subsample": 0.7,
        "subsample_freq": 1,
    },

    {
        "name": "S06_feature08_row08",
        "colsample_bytree": 0.8,
        "subsample": 0.8,
        "subsample_freq": 1,
    },

    {
        "name": "S07_feature07_row08",
        "colsample_bytree": 0.7,
        "subsample": 0.8,
        "subsample_freq": 1,
    },
]

In [76]:
sampling_results = []

for trial in sampling_trials:

    name = trial["name"]

    params = {
        **tuned_base_params,
        **{
            k: v
            for k, v in trial.items()
            if k != "name"
        },
    }

    result = evaluate_lgbm_params(
        params,
        cv_data,
    )

    sampling_results.append({
        "experiment": name,
        "colsample_bytree": params["colsample_bytree"],
        "subsample": params["subsample"],
        "subsample_freq": params["subsample_freq"],
        "mean_cv_auc": result["mean_auc"],
        "std_cv_auc": result["std_auc"],
        "mean_best_iteration": result["mean_best_iteration"],
    })

    print(
        f"{name}: "
        f"{result['mean_auc']:.6f} "
        f"+/- {result['std_auc']:.6f}"
    )

S01_no_sampling: 0.782863 +/- 0.003184
S02_feature_08: 0.783620 +/- 0.003472
S03_feature_07: 0.784068 +/- 0.003413
S04_row_08: 0.785166 +/- 0.003619
S05_row_07: 0.784755 +/- 0.003347
S06_feature08_row08: 0.784852 +/- 0.003414
S07_feature07_row08: 0.784925 +/- 0.003088


In [80]:
sampling_df = (
    pd.DataFrame(sampling_results)
    .sort_values("mean_cv_auc", ascending=False)
)

BEST_COMPLEXITY_AUC = 0.783620

sampling_df["gain_vs_T04"] = (
    sampling_df["mean_cv_auc"] - BEST_COMPLEXITY_AUC
)

display(sampling_df)

,experiment,colsample_bytree,subsample,subsample_freq,mean_cv_auc,std_cv_auc,mean_best_iteration,gain_vs_T04
3,S04_row_08,1.0,0.8,1,0.785166,0.003619,1217.2,1.546130e-03
6,S07_feature07_row08,0.7,0.8,1,0.784925,0.003088,1240.4,1.304642e-03
5,S06_feature08_row08,0.8,0.8,1,0.784852,0.003414,1185.4,1.231786e-03
4,S05_row_07,1.0,0.7,1,0.784755,0.003347,1142.0,1.135204e-03
2,S03_feature_07,0.7,1.0,0,0.784068,0.003413,1173.6,4.483664e-04
1,S02_feature_08,0.8,1.0,0,0.783620,0.003472,1140.4,9.318452e-08
0,S01_no_sampling,1.0,1.0,0,0.782863,0.003184,1168.8,-7.569705e-04


# Final modeling

In [67]:
final_params = {
    # fixed base settings (never tuned)
    "objective": "binary",
    "n_estimators": 1200,      # ceiling; early stopping decides the real count
    "learning_rate": 0.03,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,

    # T-stage winner (T04)
    "num_leaves": 31,
    "min_child_samples": 50,

    # S-stage winner (S04_row_08)
    "colsample_bytree": 1.0,
    "subsample": 0.8,
    "subsample_freq": 1,
}
BEST_PARAMS = dict(
    objective="binary",
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=1.0,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

In [68]:
FINAL_FEATURES = X.columns.tolist()

X_final = X[FINAL_FEATURES].copy()
X_test_final = X_test[FINAL_FEATURES].copy()

print("Training features:", X_final.shape)
print("Test features:", X_test_final.shape)

Training features: (307511, 201)
Test features: (48744, 201)


In [69]:
final_preprocessor = make_tree_preprocessor(X_final)

X_final_t = final_preprocessor.fit_transform(X_final)
X_test_final_t = final_preprocessor.transform(X_test_final)

print("Transformed train shape:", X_final_t.shape)
print("Transformed test shape:", X_test_final_t.shape)

The history saving thread hit an unexpected error (OperationalError('unable to open database file')).History will not be written to the database.
Transformed train shape: (307511, 331)
Transformed test shape: (48744, 331)


In [54]:
from sklearn.model_selection import train_test_split

X_tr, X_es, y_tr, y_es = train_test_split(
    X_final_t, y, test_size=0.1, stratify=y, random_state=42
)

final_model = LGBMClassifier(**final_params)
final_model.fit(
    X_tr, y_tr,
    eval_set=[(X_es, y_es)],
    eval_metric="auc",
    callbacks=[early_stopping(200, verbose=False), log_evaluation(0)],
)

print(f"Best iteration: {final_model.best_iteration_}")

.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Best iteration: 764


In [61]:
final_model = LGBMClassifier(**final_params)

final_model.fit(
    X_final_t,
    y,
    callbacks=[
        log_evaluation(0)
    ],
)

,learning_rate,0.03
,n_estimators,5000
,objective,'binary'
,min_child_samples,50
,subsample,0.8
,subsample_freq,1
,reg_alpha,0.1
,reg_lambda,0.1
,random_state,42
,n_jobs,-1
,verbosity,-1


In [49]:
final_params["n_estimators"]

5000

In [70]:
final_model = LGBMClassifier(**final_params)

final_model.fit(
    X_final_t,
    y,
    callbacks=[
        log_evaluation(0)
    ],
)

,learning_rate,0.03
,n_estimators,1200
,objective,'binary'
,min_child_samples,50
,subsample,0.8
,subsample_freq,1
,reg_alpha,0.1
,reg_lambda,0.1
,random_state,42
,n_jobs,-1
,verbosity,-1


In [71]:
test_pred = final_model.predict_proba(
    X_test_final_t
)[:, 1]

In [72]:
print("Prediction count:", len(test_pred))
print("Min prediction:", test_pred.min())
print("Max prediction:", test_pred.max())
print("Mean prediction:", test_pred.mean())

Prediction count: 48744
Min prediction: 0.0019425297035048678
Max prediction: 0.8519240765551952
Mean prediction: 0.07387531402238172


In [73]:
submission = pd.DataFrame({
    "SK_ID_CURR": test["SK_ID_CURR"],
    "TARGET": test_pred
})

display(submission.head())
print(submission.shape)

,SK_ID_CURR,TARGET
0,100001,0.030448
1,100005,0.153526
2,100013,0.020219
3,100028,0.035504
4,100038,0.188438


(48744, 2)


In [74]:
assert submission.shape[0] == test.shape[0]
assert submission["SK_ID_CURR"].notna().all()
assert submission["TARGET"].notna().all()
assert submission["TARGET"].between(0, 1).all()

In [75]:
submission.to_csv(
    "submission.csv",
    index=False
)

print("Saved submission.csv")

Saved submission.csv


# CatBoost

In [78]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

In [79]:
X_cat = X_final.copy()
X_test_cat = X_test_final.copy()

cat_cols = X_cat.select_dtypes(exclude=np.number).columns.tolist()

for col in cat_cols:
    X_cat[col] = X_cat[col].fillna("MISSING").astype(str)
    X_test_cat[col] = X_test_cat[col].fillna("MISSING").astype(str)

cat_feature_indices = [X_cat.columns.get_loc(col) for col in cat_cols]

print(f"CatBoost feature count: {X_cat.shape[1]}")
print(f"Categorical feature count: {len(cat_cols)}")
print(cat_cols)

CatBoost feature count: 201
Categorical feature count: 16
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


In [80]:
cat_oof = np.zeros(len(X_cat))
cat_test_pred = np.zeros(len(X_test_cat))
cat_fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X_cat, y), start=1):
    X_train_cat = X_cat.iloc[train_idx]
    y_train_cat = y.iloc[train_idx]

    X_valid_cat = X_cat.iloc[valid_idx]
    y_valid_cat = y.iloc[valid_idx]

    cat_model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=5000,
        learning_rate=0.03,
        depth=6,
        l2_leaf_reg=5.0,
        random_seed=42 + fold,
        random_strength=1.0,
        border_count=128,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )

    cat_model.fit(
        X_train_cat,
        y_train_cat,
        cat_features=cat_feature_indices,
        eval_set=(X_valid_cat, y_valid_cat),
        early_stopping_rounds=200,
        verbose=False,
    )

    valid_pred = cat_model.predict_proba(X_valid_cat)[:, 1]
    cat_oof[valid_idx] = valid_pred

    fold_auc = roc_auc_score(y_valid_cat, valid_pred)
    cat_fold_scores.append(fold_auc)

    cat_test_pred += (
        cat_model.predict_proba(X_test_cat)[:, 1] / cv.n_splits
    )

    print(
        f"Fold {fold}: AUC = {fold_auc:.5f}, "
        f"best iteration = {cat_model.get_best_iteration()}"
    )

print(f"\nCatBoost mean CV AUC: {np.mean(cat_fold_scores):.5f}")
print(f"CatBoost std CV AUC: {np.std(cat_fold_scores):.5f}")
print(f"CatBoost OOF AUC: {roc_auc_score(y, cat_oof):.5f}")

Fold 1: AUC = 0.78222, best iteration = 4146
Fold 2: AUC = 0.79012, best iteration = 3887
Fold 3: AUC = 0.78372, best iteration = 3836
Fold 4: AUC = 0.78958, best iteration = 4508
Fold 5: AUC = 0.78181, best iteration = 2936

CatBoost mean CV AUC: 0.78549
CatBoost std CV AUC: 0.00362
CatBoost OOF AUC: 0.78548


In [83]:
oof_lgbm_s04 = np.zeros(len(X))
test_lgbm_s04 = np.zeros(len(X_test))
lgbm_s04_fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    fold_preprocessor = make_tree_preprocessor(X_train)

    X_train_t = fold_preprocessor.fit_transform(X_train)
    X_valid_t = fold_preprocessor.transform(X_valid)
    X_test_t = fold_preprocessor.transform(X_test)

    lgbm_s04_model = LGBMClassifier(**final_params)

    lgbm_s04_model.fit(
        X_train_t,
        y_train,
        callbacks=[log_evaluation(0)],
    )

    valid_pred = lgbm_s04_model.predict_proba(X_valid_t)[:, 1]
    test_fold_pred = lgbm_s04_model.predict_proba(X_test_t)[:, 1]

    oof_lgbm_s04[valid_idx] = valid_pred
    test_lgbm_s04 += test_fold_pred / cv.n_splits

    fold_auc = roc_auc_score(y_valid, valid_pred)
    lgbm_s04_fold_scores.append(fold_auc)

    print(f"Fold {fold}: AUC = {fold_auc:.5f}")

print(f"\nLightGBM S04 mean CV AUC: {np.mean(lgbm_s04_fold_scores):.5f}")
print(f"LightGBM S04 OOF AUC: {roc_auc_score(y, oof_lgbm_s04):.5f}")

Fold 1: AUC = 0.78087
Fold 2: AUC = 0.79006
Fold 3: AUC = 0.78319
Fold 4: AUC = 0.78831
Fold 5: AUC = 0.78198

LightGBM S04 mean CV AUC: 0.78488
LightGBM S04 OOF AUC: 0.78487


In [ ]:
oof_prediction_correlation = np.corrcoef(
    oof_lgbm_s04,
    cat_oof,
)[0, 1]

print(f"OOF prediction correlation: {oof_prediction_correlation:.5f}")

In [84]:
blend_results = []

for lgbm_weight in np.arange(0.0, 1.01, 0.05):
    catboost_weight = 1.0 - lgbm_weight

    blended_oof = (
        lgbm_weight * oof_lgbm_s04
        + catboost_weight * cat_oof
    )

    blend_auc = roc_auc_score(y, blended_oof)

    blend_results.append({
        "lgbm_weight": round(lgbm_weight, 2),
        "catboost_weight": round(catboost_weight, 2),
        "oof_auc": blend_auc,
    })

blend_results_df = (
    pd.DataFrame(blend_results)
    .sort_values("oof_auc", ascending=False)
    .reset_index(drop=True)
)

display(blend_results_df.head(10))

,lgbm_weight,catboost_weight,oof_auc
0,0.45,0.55,0.787171
1,0.50,0.50,0.787153
2,0.40,0.60,0.787149
3,0.55,0.45,0.787098
4,0.35,0.65,0.787088
5,0.60,0.40,0.787005
6,0.30,0.70,0.786988
7,0.65,0.35,0.786873
8,0.25,0.75,0.786847
9,0.70,0.30,0.786704


In [85]:
best_lgbm_weight = blend_results_df.loc[0, "lgbm_weight"]
best_catboost_weight = blend_results_df.loc[0, "catboost_weight"]

blend_test_pred = (
    best_lgbm_weight * test_lgbm_s04
    + best_catboost_weight * cat_test_pred
)

submission_blend = pd.DataFrame({
    "SK_ID_CURR": test_ids,
    "TARGET": blend_test_pred,
})

submission_blend.to_csv(
    "submission_lgbm_catboost_blend.csv",
    index=False,
)

print(submission_blend.shape)
display(submission_blend.head())

(48744, 2)


,SK_ID_CURR,TARGET
0,100001,0.034655
1,100005,0.170903
2,100013,0.015501
3,100028,0.032976
4,100038,0.148470
